# 01 · Published checkpoints on dev, test and the no-speech clips

Runs on Kaggle with a GPU. **Settings → Accelerator: GPU, Internet: On.** No manual uploads: the
notebook clones the public repository, which contains the dataset (`data/`).

Every system in `configs/baselines.yaml` with `trained_by_us: false` and `status: planned` is run by
`scripts/infer_pretrained.py`, one process per system. Checkpoints are pinned to a commit SHA
(written to `results/predictions/<system>.meta.json`). Then `scripts/score.py` writes the tables.

`TRIAL = True` runs 8 clips per split into `/kaggle/working/trial` (a short test of every system).

Output: `/kaggle/working/results_baselines.zip` with `results/`. Unzip it at the repository root.

In [ ]:
REPO_URL = "https://github.com/eneserman/NLP.git"  # the public repository
BRANCH = "main"
TRIAL = False
BATCH_SIZE = 16

In [ ]:
import os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN loaded")
except Exception:
    print("No HF_TOKEN secret; using anonymous access")

In [ ]:
%cd /kaggle/working
!rm -rf repo && git clone -q -b $BRANCH $REPO_URL repo
%cd /kaggle/working/repo
!git log -1 --format='commit %H'
!pip install -q -r requirements/baselines.txt
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip freeze | grep -iE '^(torch|torchaudio|transformers|nemo[_-]toolkit|huggingface[_-]hub|numpy|pyyaml|lightning|jiwer|pandas|pyarrow)=='

Tests of the inference code (they use the real torch and transformers here).

In [ ]:
!python -m pytest -q tests/test_inference.py

In [ ]:
!python scripts/unpack_audio.py

In [ ]:
import subprocess, sys, yaml
listed = yaml.safe_load(open("configs/baselines.yaml", encoding="utf-8"))["baselines"]
systems = [b["system"] for b in listed if not b.get("trained_by_us") and b.get("status") == "planned"]
print(systems)
extra = ["--limit", "8", "--out-dir", "/kaggle/working/trial"] if TRIAL else []
failed = []
for s in systems:
    print("=" * 30, s, flush=True)
    r = subprocess.run([sys.executable, "scripts/infer_pretrained.py", "--system", s,
                        "--batch-size", str(BATCH_SIZE), *extra])
    if r.returncode != 0:
        failed.append(s)
print("failed:", failed)

In [ ]:
if TRIAL:
    !head -n 3 /kaggle/working/trial/*.test.jsonl
    !cat /kaggle/working/trial/*.meta.json
else:
    !python scripts/score.py --split dev
    !python scripts/score.py --split test
    !zip -q -r /kaggle/working/results_baselines.zip results && ls -lh /kaggle/working/results_baselines.zip